In [2]:
!pip install -q transformers accelerate torch

In [3]:
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
texts = [
    "The transformer model achieved excellent accuracy.",
    "Large Language Models are revolutionizing AI.",
    "The football team won the championship.",
    "The cricket match was exciting.",
    "Neural networks are widely used in deep learning.",
    "The player scored a brilliant goal.",
    "Machine learning improves decision making.",
    "The tennis tournament starts tomorrow."
]

labels = [
    1, 1, 0, 0,
    1, 0, 1, 0
]

# 0 = Sports
# 1 = Technology

print("Dataset created successfully!")
print("Number of examples:", len(texts))

Dataset created successfully!
Number of examples: 8


In [5]:
model_name = "bert-base-uncased"

print("Loading BERT model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("BERT model loaded successfully!")

Loading BERT model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded successfully!


In [6]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed successfully!")

Tokenization completed successfully!


In [8]:
class DomainDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

    def __len__(self):
        return len(self.labels)


dataset = DomainDataset(
    encodings,
    labels
)

print("PyTorch dataset created successfully!")

PyTorch dataset created successfully!


In [9]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

print("Starting fine-tuning...")

trainer.train()

print("Fine-tuning completed successfully!")

Starting fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.884344
2,0.561952
3,0.686502
4,0.532372
5,0.497187
6,0.540128
7,0.514372
8,0.433263
9,0.491735
10,0.493317


Fine-tuning completed successfully!


In [11]:
trainer.save_model("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Fine-tuned model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!


In [12]:
classifier = pipeline(
    "text-classification",
    model="./fine_tuned_model",
    tokenizer="./fine_tuned_model"
)

text = "Generative AI models improve intelligent automation."

result = classifier(text)

labels_map = {
    "LABEL_0": "Sports",
    "LABEL_1": "Technology"
}

print("\nPrediction")
print("-------------------------")
print("Input :", text)
print(
    "Predicted Class :",
    labels_map[result[0]["label"]]
)
print(
    "Confidence Score :",
    round(result[0]["score"], 3)
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Prediction
-------------------------
Input : Generative AI models improve intelligent automation.
Predicted Class : Technology
Confidence Score : 0.723
